In [1]:
import os
import json
import time
import re
import chromadb
import numpy as np

from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"]
)

MODEL = "meta/llama-3.1-70b-instruct"

In [3]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

collection = chromadb_client.get_collection(
    "tesla-10k-2019-to-2023"
)

data = collection.get()

documents = data["documents"]
chunk_ids = data["ids"]
metadatas = data["metadatas"]

tesla_chunks = [
    {
        "chunk_id": chunk_ids[i],
        "chunk_text": documents[i],
        "metadata": metadatas[i]
    }
    for i in range(len(documents))
]

print("Total chunks:", len(tesla_chunks))

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Total chunks: 3337


compression function

Compression in your code does not reduce the number of chunks.

1 chunk

↓

Keeps only important sentences

↓

Returns compressed text

In [4]:
def compress_chunk(text, top_n=8):

    text = text.replace("\t", " ")
    text = text.replace("\n", " ")

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    sentences = [
        s.strip()
        for s in sentences
        if len(s.strip()) > 30
    ]

    if len(sentences) <= top_n:
        return text

    try:

        vectorizer = TfidfVectorizer(
            stop_words="english"
        )

        X = vectorizer.fit_transform(
            sentences
        )

        scores = np.asarray(
            X.sum(axis=1)
        ).flatten()

        top_indices = scores.argsort()[-top_n:]

        return " ".join(
            sentences[i]
            for i in sorted(top_indices)
        )

    except:
        return text

skip noisy Chunks

In [5]:
def skip_chunk(text):

    text = text.lower()

    useless_patterns = [
        "table of contents",
        "signature",
        "index",
        "auditor"
    ]

    return any(
        p in text
        for p in useless_patterns
    )

In [6]:
system_prompt = """
You are an expert financial retrieval engineer.

You will receive approximately 100 Tesla 10-K chunks.

Generate EXACTLY 3 hypothetical questions that can be answered using information contained somewhere in these chunks.

The questions should:

- represent important themes
- be useful for retrieval
- cover risks, finance, operations, technology, governance, strategy, regulations
- not hallucinate information
- not mention chunk numbers

Output exactly:

Question 1
Question 2
Question 3

Nothing else.
"""

In [7]:
def call_nvidia(prompt):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()

batch prompt builder

In [8]:
def create_batch_prompt(batch):

    text = ""

    for item in batch:

        if skip_chunk(
            item["chunk_text"]
        ):
            continue

        compressed_text = compress_chunk(
            item["chunk_text"]
        )

        text += compressed_text
        text += "\n\n"

    return text

parse questions

In [9]:
def parse_questions(response_text):

    questions = []

    for line in response_text.splitlines():

        line = line.strip()

        if line:
            questions.append(line)

    return questions[:3]

output file

In [10]:
OUTPUT_FILE = "batch_hypothetical_questions.json"

if os.path.exists(OUTPUT_FILE):

    with open(
        OUTPUT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        output_data = json.load(f)

else:

    output_data = []

checkpoint

In [11]:
CHECKPOINT_FILE = "batch_checkpoint.json"

batch_size = 100

if os.path.exists(CHECKPOINT_FILE):

    with open(
        CHECKPOINT_FILE,
        "r"
    ) as f:

        checkpoint = json.load(f)

        last_start = checkpoint.get(
            "last_start",
            -batch_size
        )

else:

    last_start = -batch_size

main loop

In [ ]:
# for start in range(
#     last_start + batch_size,
#     len(tesla_chunks),
#     batch_size
# ):

#     batch = tesla_chunks[
#         start:start + batch_size
#     ]

#     try:

#         print(
#             f"\nProcessing batch "
#             f"{start} - "
#             f"{start + len(batch)-1}"
#         )

#         prompt = create_batch_prompt(
#             batch
#         )

#         response_text = call_nvidia(
#             prompt
#         )

#         questions = parse_questions(
#             response_text
#         )

#         batch_record = {

#             "batch_id":
#                 start // batch_size,

#             "start_chunk":
#                 start,

#             "end_chunk":
#                 start + len(batch) - 1,

#             "chunk_ids": [
#                 chunk["chunk_id"]
#                 for chunk in batch
#             ],

#             "hypothetical_questions":
#                 questions
#         }

#         output_data.append(
#             batch_record
#         )

#         with open(
#             OUTPUT_FILE,
#             "w",
#             encoding="utf-8"
#         ) as f:

#             json.dump(
#                 output_data,
#                 f,
#                 indent=2,
#                 ensure_ascii=False
#             )

#         with open(
#             CHECKPOINT_FILE,
#             "w"
#         ) as f:

#             json.dump(
#                 {
#                     "last_start": start
#                 },
#                 f
#             )

#         print(
#             "Questions generated:"
#         )

#         for q in questions:
#             print("-", q)

#         time.sleep(2)

#     except Exception as e:

#         print("Error:", e)

#         if "rate_limit" in str(e).lower():

#             print(
#                 "Rate limited. Waiting..."
#             )

#             time.sleep(20)

#             continue

#         break


Processing batch 0 - 99
Questions generated:
- Question 1: What is the total number of shares underlying option awards outstanding for each of Tesla's non-employee directors as of December 31, 2021?
- Question 2: How does Tesla's compensation program for its non-employee directors align with its overall compensation philosophy, and what types of compensation do directors receive?
- Question 3: What is the purpose of the 2018 CEO Performance Award granted to Elon Musk, and how does it align with Tesla's long-term goals and stockholder interests?

Processing batch 100 - 199
Questions generated:
- Question 1: What is the maximum amount of transferable tax credits that Tesla was eligible for under the agreements with the State of Nevada and Storey County in Nevada in connection with the construction of Gigafactory Nevada?
- Question 2: What is the warranty period for the battery and drive unit on Tesla's current new Model 3 and Model Y vehicles, and what is the minimum retention of batter

In [17]:
import json

with open(
    "batch_hypothetical_questions.json",
    "r",
    encoding="utf-8"
) as f:
    output_data = json.load(f)

print(len(output_data))

34


In [18]:
from langchain_core.documents import Document

hypothetical_documents = []

for batch in output_data:

    for question_text in batch["hypothetical_questions"]:

        hypothetical_documents.append(
            Document(
                page_content=question_text,
                metadata={
                    "batch_id": batch["batch_id"],
                    "start_chunk": batch["start_chunk"],
                    "end_chunk": batch["end_chunk"],
                    "chunk_ids": ",".join(batch["chunk_ids"])
                }
            )
        )

print(
    "Total HQ Documents:",
    len(hypothetical_documents)
)

Total HQ Documents: 102


create a new collection inside the same tesla_db

In [19]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

hypothetical_vectorstore = Chroma(
    collection_name="tesla_hypothetical_questions",
    embedding_function=embedding_function,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [20]:
hypothetical_vectorstore.add_documents(
    hypothetical_documents
)

['73d71e51-af49-4b4e-acb0-f177a8ea34d5',
 '10bfac06-18a1-42b2-9b1b-ccb7a2ae676e',
 'e0d2a82d-df78-442d-8e10-b90f9eb458ce',
 '4bb4ef29-d9cd-4c0f-a53a-caea756f2f63',
 'd9ba27e7-e12b-40ec-a9b0-6fa0775e7826',
 '1079c6f2-dc83-49fd-af19-30e8061b434a',
 'ef8ecd13-457a-40b5-ae14-c17ff2c5d7f6',
 'f8925bfa-1e7a-4b86-abb8-fea83da5ee6d',
 'de7448ff-b6e3-419b-a426-a5d23399eb0e',
 '377e5dad-40a5-475a-a44c-85f849d34c5d',
 '7d5763b3-b85e-42a3-9eee-f34efe1e3a00',
 '13dd0d46-64a1-4a62-bc2d-57ad35d6d2e5',
 '110cc737-2575-4dcf-a3fe-4cdcbe887f7c',
 '0bfad92e-7328-433c-ad2f-c530f365c914',
 'b194fac0-4d23-487e-b97e-17a17aa46770',
 '3233e6d3-a6d9-4ee8-b714-48372b377200',
 'eb03fb66-5378-4eaa-a1f6-0d08ec846334',
 'c37ecaa8-64f7-4b18-9592-ccb6400315b6',
 'b5c6a95d-9885-4109-9eab-43f383f3a1b9',
 '99c94c35-6e6c-4d3f-b180-74512df65ea8',
 '36bddaa8-c337-4f7a-86cd-51870b402269',
 'cbf479ec-2192-41fa-a06c-8f32de18b8b2',
 '9e32af95-8be1-4c94-82c8-d2ad7ca9c82d',
 'e35f373f-328f-4e69-a92e-dfa58a571404',
 'fd11ded3-74fc-

In [21]:
chromadb_client.list_collections()

['tesla_hypothetical_questions', 'tesla-10k-2019-to-2023']

In [22]:
hq_collection = chromadb_client.get_collection(
    "tesla_hypothetical_questions"
)

print(
    hq_collection.count()
)

102


Hypothetical Retriever

In [24]:
hq_retriever = hypothetical_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [27]:
benchmark_questions = {
    "HQ1": "What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",

    "HQ2": "How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",

    "HQ3": "What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",

    "HQ4": "Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?"
}

In [31]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [33]:
results = []

for question_id, user_query in benchmark_questions.items():

    print(f"\nRunning {question_id}")

    retrieved_hqs = hq_retriever.invoke(user_query)

    results.append({
        "question_id": question_id,
        "user_query": user_query,
        "retrieved_hqs": retrieved_hqs
    })


Running HQ1

Running HQ2

Running HQ3

Running HQ4


extract relevant tesla chunk IDs

In [34]:
metadata = {
    "batch_id": ...,
    "start_chunk": ...,
    "end_chunk": ...,
    "chunk_ids": "text_0,text_1,text_2..."
}

In [35]:
for result in results:

    parent_chunk_ids = set()

    for doc in result["retrieved_hqs"]:

        ids = doc.metadata["chunk_ids"].split(",")

        parent_chunk_ids.update(ids)

    result["parent_chunk_ids"] = list(parent_chunk_ids)

In [36]:
for result in results:

    parent_chunks = []

    for chunk_id in result["parent_chunk_ids"]:

        chunk_data = collection.get(
            ids=[chunk_id]
        )

        if len(chunk_data["documents"]) > 0:

            parent_chunks.append({
                "chunk_id": chunk_id,
                "text": chunk_data["documents"][0]
            })

    result["parent_chunks"] = parent_chunks

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEv

Build Context, so it doesnt send all chunks to LLM

In [37]:
MAX_CHUNKS = 20

for result in results:

    context = "\n\n".join(
        chunk["text"]
        for chunk in result["parent_chunks"][:MAX_CHUNKS]
    )

    result["context"] = context

Generate Answer

In [38]:
for result in results:

    prompt = [
        {
            "role": "system",
            "content": qna_system_message
        },
        {
            "role": "user",
            "content": qna_user_message_template.format(
                context=result["context"],
                question=result["user_query"]
            )
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=prompt,
        temperature=0.2,
        max_tokens=1000
    )

    result["final_answer"] = (
        response.choices[0]
        .message
        .content
    )

In [42]:
for result in assignment_outputs:

    print("\n" + "="*80)

    print("Question ID:", result["question_id"])

    print("\nQuery:")
    print(result["user_query"])

    print("\nAnswer:")
    print(result["final_answer"])


Question ID: HQ1

Query:
What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?

Answer:
A board member should ask about the following risks that could prevent Tesla from meeting production, delivery, or scaling expectations:

1. Delays in establishing and/or sustaining vehicle ramps, particularly for Model 3 and Model Y.
2. Issues with ongoing manufacturing process improvements and cost-down efforts.
3. Challenges in hiring, training, and compensating skilled employees to operate high-volume production facilities.
4. Potential bottlenecks in production due to shared parts, suppliers, or production facilities among different models.
5. Delays in launching and/or ramping production of new vehicles, such as Tesla Semi, Cybertruck, and the new Tesla Roadster.
6. Challenges in designing, constructing, and obtaining regulatory approvals for future manufacturing facilities, including Gigafactory Berlin.
7. Inaccurate f

In [43]:
assignment_outputs = []

for result in results:

    assignment_outputs.append({

        "question_id":
            result["question_id"],

        "user_query":
            result["user_query"],

        "retrieved_hypothetical_questions":[
            {
                "hypothetical_question":
                    doc.page_content,

                "batch_id":
                    doc.metadata["batch_id"]
            }
            for doc in result["retrieved_hqs"]
        ],

        "parent_chunks_used":[
            {
                "chunk_id":
                    chunk["chunk_id"]
            }
            for chunk in result["parent_chunks"][:20]
        ],

        "final_answer":
            result["final_answer"],

        "citations":[
            chunk["chunk_id"]
            for chunk in result["parent_chunks"][:20]
        ]
    })

In [44]:
with open(
    "outputs.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        assignment_outputs,
        f,
        indent=2,
        ensure_ascii=False
    )